# 🛡️ DeepShield V3-Max — Multi-Generator Training (Kaggle edition)

Same recipe as the Colab notebook, adapted for **Kaggle Notebooks**: 30 GPU-hours/week, 12-hour sessions, and every dataset lives on Kaggle already.

## Setup — just two switches

Right panel → **Session options**:
- **Accelerator** → **GPU T4 x2** (or P100)
- **Internet** → **On**

Then **Run All**. That is enough — the notebook downloads any dataset it needs.

*(Optional speed-up: attaching the datasets via **+ Add Input** skips the downloads. The notebook uses attached copies automatically when it finds them, so this is purely optional.)*

Checkpoints are written to `/kaggle/working/`; the trained model appears in the **Output** tab on the right.

In [ ]:
# ── 1. Environment ───────────────────────────────────────────────
import torch, os, glob
print('PyTorch :', torch.__version__)
assert torch.cuda.is_available(), 'No GPU! Session options → Accelerator → GPU'
device = torch.device('cuda')
print('GPU     :', torch.cuda.get_device_name(0))

# /kaggle/working persists for the whole session (and into saved versions)
CKPT_DIR = '/kaggle/working'
RESUME_PATH = f'{CKPT_DIR}/resume.pth'
print('Checkpoints →', CKPT_DIR)

In [ ]:
# ── 2. Get datasets: use attached inputs, else auto-download ─────
# Attaching via "+ Add Input" is fastest (no download), but if a set
# is not attached we fall back to kagglehub (needs Internet: On).
INPUT = '/kaggle/input'
attached = sorted(os.listdir(INPUT)) if os.path.isdir(INPUT) else []
print('Attached inputs:', attached or 'none — will download instead')

def find_root(base, marker_dirs):
    for root, dirs, _ in os.walk(base):
        if marker_dirs <= set(dirs):
            return root
    return None

def all_images(path):
    files = []
    for e in ('*.jpg', '*.jpeg', '*.png', '*.webp'):
        files += glob.glob(os.path.join(path, '**', e), recursive=True)
    return sorted(files)

def input_dir(*keywords):
    for name in attached:
        low = name.lower()
        if all(k in low for k in keywords):
            return os.path.join(INPUT, name)
    return None

def get_dataset(slug, *keywords):
    """Attached folder if present, otherwise download it."""
    d = input_dir(*keywords)
    if d:
        return d, 'attached'
    import kagglehub
    return kagglehub.dataset_download(slug), 'downloaded'

# -- Base: 140k real-and-fake (SG1 fakes + FFHQ reals) — REQUIRED
base, how = get_dataset('xhlulu/140k-real-and-fake-faces', '140k')
DATA = find_root(base, {'train', 'valid', 'test'})
assert DATA, 'train/valid/test folders not found in the 140k dataset'
print(f'140k base   : {DATA}  ({how})')

# -- Extra fake families (tpdn required; others optional)
EXTRAS = {}
for key, slug, keywords in [
    ('tpdn',      'almightyj/person-face-dataset-thispersondoesnotexist', ('thispersondoesnotexist',)),
    ('sg2_hires', 'hyperclaw79/fakefaces',                                ('fakefaces',)),
    ('diffusion', 'mohannadaymansalah/stable-diffusion-dataaaaaaaaa',     ('diffusion',)),
]:
    try:
        d, how = get_dataset(slug, *keywords)
        imgs = all_images(d)
        EXTRAS[key] = imgs
        print(f'{key:10s}: {len(imgs):6d} images  ({how})')
    except Exception as e:
        print(f'{key:10s}: SKIPPED ({type(e).__name__})')
assert EXTRAS.get('tpdn'), 'TPDN set unavailable — check Internet: On in Session options'

In [ ]:
# ── 3. Configuration ─────────────────────────────────────────────
BACKBONE          = 'large'   # 'large' (V3-Max) or 'small' (V2-style)
FAKE_TOTAL        = 50000     # per-class budget (matches available reals)
VAL_PER_CLASS     = 2500
TPDN_HOLDOUT_N    = 1000      # never trained on — the revenge metric
EPOCHS            = 10
BATCH             = 128
LR                = 3e-4
IMG               = 224
SEED              = 42

import random, numpy as np
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

In [ ]:
# ── 4. Build the multi-generator training set ────────────────────
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image as PILImage
import io

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

class RandomJPEG:
    def __init__(self, p=0.7, quality=(30, 95)):
        self.p, self.quality = p, quality
    def __call__(self, img):
        if random.random() < self.p:
            buf = io.BytesIO()
            img.save(buf, 'JPEG', quality=random.randint(*self.quality))
            buf.seek(0)
            img = PILImage.open(buf).convert('RGB')
        return img

class RandomRescale:
    def __init__(self, p=0.5, lo=0.5):
        self.p, self.lo = p, lo
    def __call__(self, img):
        if random.random() < self.p:
            w, h = img.size
            s = random.uniform(self.lo, 1.0)
            img = img.resize((max(32, int(w*s)), max(32, int(h*s))), PILImage.BILINEAR)
            img = img.resize((w, h), PILImage.BILINEAR)
        return img

class FixedJPEG:
    def __init__(self, quality=40):
        self.q = quality
    def __call__(self, img):
        buf = io.BytesIO()
        img.save(buf, 'JPEG', quality=self.q)
        buf.seek(0)
        return PILImage.open(buf).convert('RGB')

train_tf = transforms.Compose([
    RandomRescale(), RandomJPEG(),
    transforms.RandomResizedCrop(IMG, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
    transforms.RandomGrayscale(p=0.05),
    transforms.RandomApply([transforms.GaussianBlur(3)], p=0.2),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD),
])
eval_tf   = transforms.Compose([transforms.Resize((IMG, IMG)), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
robust_tf = transforms.Compose([FixedJPEG(40), transforms.Resize((IMG, IMG)), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

class FlatImages(Dataset):
    """List of (path, label) with a transform. label: 0=fake, 1=real."""
    def __init__(self, samples, tf):
        self.samples, self.tf = samples, tf
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, i):
        p, y = self.samples[i]
        img = PILImage.open(p).convert('RGB')
        return self.tf(img), y

CLASSES = ['fake', 'real']  # index order MUST match the app
rng = random.Random(SEED)

# --- fakes: TPDN holdout carved out FIRST, then fill the budget
tpdn_all = list(EXTRAS['tpdn']); rng.shuffle(tpdn_all)
tpdn_holdout = tpdn_all[:TPDN_HOLDOUT_N]
tpdn_train   = tpdn_all[TPDN_HOLDOUT_N:]

sg2   = list(EXTRAS.get('sg2_hires', [])); rng.shuffle(sg2)
diff  = list(EXTRAS.get('diffusion', [])); rng.shuffle(diff)
sg1   = all_images(os.path.join(DATA, 'train', 'fake')); rng.shuffle(sg1)

fakes = tpdn_train + sg2 + diff
fakes = fakes[:FAKE_TOTAL]
fakes += sg1[:max(0, FAKE_TOTAL - len(fakes))]

reals = all_images(os.path.join(DATA, 'train', 'real')); rng.shuffle(reals)
reals = reals[:len(fakes)]

print(f'FAKE mix: tpdn={min(len(tpdn_train), FAKE_TOTAL)} sg2={len(sg2)} '
      f'diffusion={len(diff)} sg1-fill={len(fakes)-min(len(tpdn_train),FAKE_TOTAL)-len(sg2)-len(diff)}')
print(f'TRAIN: {len(fakes)} fake + {len(reals)} real | TPDN holdout: {len(tpdn_holdout)}')

train_samples = [(p, 0) for p in fakes] + [(p, 1) for p in reals]
rng.shuffle(train_samples)

def per_class_subset(folder, n, label, tf_seed):
    files = all_images(folder)
    random.Random(tf_seed).shuffle(files)
    return [(p, label) for p in files[:n]]

valid_samples = (per_class_subset(f'{DATA}/valid/fake', VAL_PER_CLASS, 0, SEED)
               + per_class_subset(f'{DATA}/valid/real', VAL_PER_CLASS, 1, SEED))
test_samples  = ([(p, 0) for p in all_images(f'{DATA}/test/fake')]
               + [(p, 1) for p in all_images(f'{DATA}/test/real')])
tpdn_samples  = [(p, 0) for p in tpdn_holdout]

train_dl  = DataLoader(FlatImages(train_samples, train_tf), batch_size=BATCH, shuffle=True,
                       num_workers=4, pin_memory=True, persistent_workers=True)
valid_dl  = DataLoader(FlatImages(valid_samples, eval_tf),   batch_size=BATCH, num_workers=2, pin_memory=True)
robust_dl = DataLoader(FlatImages(valid_samples, robust_tf), batch_size=BATCH, num_workers=2, pin_memory=True)
tpdn_dl   = DataLoader(FlatImages(tpdn_samples, eval_tf),    batch_size=BATCH, num_workers=2, pin_memory=True)
test_dl   = DataLoader(FlatImages(test_samples, eval_tf),    batch_size=BATCH, num_workers=2, pin_memory=True)

In [ ]:
# ── 5. Model: MobileNetV3 (large or small) ───────────────────────
from torchvision import models

def build_model(backbone):
    if backbone == 'large':
        m = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V2)
    else:
        m = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
    m.classifier[3] = torch.nn.Linear(m.classifier[3].in_features, 2)
    return m

model = build_model(BACKBONE).to(device)
params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'MobileNetV3-{BACKBONE} ready — {params:.1f}M parameters')

In [ ]:
# ── 6. Train — with Drive-resume + live TPDN metric ──────────────
import copy
from tqdm.auto import tqdm

criterion = torch.nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS * len(train_dl), eta_min=1e-5)

@torch.no_grad()
def evaluate(dl):
    model.eval()
    correct = total = 0
    for xb, yb in dl:
        xb, yb = xb.to(device), yb.to(device)
        correct += (model(xb).argmax(1) == yb).sum().item()
        total += yb.size(0)
    return correct / total

history = {'loss': [], 'val_acc': [], 'robust_acc': [], 'tpdn_acc': []}
best_score, best_state, start_epoch = 0.0, None, 1

# ---- RESUME: pick up exactly where a dead session left off ----
if os.path.exists(RESUME_PATH):
    ck = torch.load(RESUME_PATH, map_location=device, weights_only=False)
    if ck.get('backbone') == BACKBONE:
        model.load_state_dict(ck['model'])
        optimizer.load_state_dict(ck['optimizer'])
        scheduler.load_state_dict(ck['scheduler'])
        history     = ck['history']
        best_score  = ck['best_score']
        best_state  = ck['best_state']
        start_epoch = ck['epoch'] + 1
        print(f'RESUMED from epoch {ck["epoch"]} (best score so far {best_score:.4f})')
    else:
        print('resume file is for a different backbone — starting fresh')

for epoch in range(start_epoch, EPOCHS + 1):
    model.train()
    running = 0.0
    bar = tqdm(train_dl, desc=f'Epoch {epoch}/{EPOCHS}')
    for xb, yb in bar:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        scheduler.step()
        running += loss.item() * yb.size(0)
        bar.set_postfix(loss=f'{loss.item():.3f}')

    epoch_loss = running / len(train_samples)
    val_acc    = evaluate(valid_dl)
    robust_acc = evaluate(robust_dl)
    tpdn_acc   = evaluate(tpdn_dl)   # ← THE revenge metric
    for k, v in [('loss', epoch_loss), ('val_acc', val_acc),
                 ('robust_acc', robust_acc), ('tpdn_acc', tpdn_acc)]:
        history[k].append(v)
    print(f'Epoch {epoch}: loss {epoch_loss:.4f} · val {val_acc*100:.2f}% '
          f'· robust {robust_acc*100:.2f}% · TPDN {tpdn_acc*100:.2f}%')

    # best = balanced robust+tpdn (both generalization axes)
    score = 0.5 * robust_acc + 0.5 * tpdn_acc
    if score > best_score:
        best_score = score
        best_state = copy.deepcopy(model.state_dict())

    torch.save({'epoch': epoch, 'backbone': BACKBONE,
                'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
                'scheduler': scheduler.state_dict(), 'history': history,
                'best_score': best_score, 'best_state': best_state}, RESUME_PATH)

model.load_state_dict(best_state)
print(f'\nBest combined score: {best_score:.4f} (0.5*robust + 0.5*TPDN)')

In [ ]:
# ── 7. Curves (report material) ──────────────────────────────────
import matplotlib.pyplot as plt
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
ep = range(1, len(history['loss']) + 1)
a1.plot(ep, history['loss'], marker='o'); a1.set_title('Training loss'); a1.set_xlabel('epoch')
a2.plot(ep, [a*100 for a in history['val_acc']],    marker='o', color='green',  label='clean val')
a2.plot(ep, [a*100 for a in history['robust_acc']], marker='s', color='orange', label='robust val (q40)')
a2.plot(ep, [a*100 for a in history['tpdn_acc']],   marker='^', color='red',    label='TPDN holdout')
a2.set_title('Accuracy (%)'); a2.set_xlabel('epoch'); a2.legend()
plt.tight_layout(); plt.savefig('training_curves.png', dpi=150); plt.show()

In [ ]:
# ── 8. Final evaluation: 140k test + TPDN holdout ────────────────
from sklearn.metrics import confusion_matrix, classification_report

test_acc = evaluate(test_dl)
tpdn_final = evaluate(tpdn_dl)
print(f'140k TEST accuracy : {test_acc*100:.2f}%')
print(f'TPDN holdout       : {tpdn_final*100:.2f}%  ← thispersondoesnotexist!')

preds, trues = [], []
with torch.no_grad():
    for xb, yb in tqdm(test_dl, desc='confusion'):
        preds += model(xb.to(device)).argmax(1).cpu().tolist()
        trues += yb.tolist()
print(classification_report(trues, preds, target_names=CLASSES))
cm = confusion_matrix(trues, preds)
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(cm, cmap='Blues')
for i in range(2):
    for j in range(2):
        ax.text(j, i, f'{cm[i, j]:,}', ha='center', va='center',
                color='white' if cm[i, j] > cm.max()/2 else 'black')
ax.set_xticks([0, 1], CLASSES); ax.set_yticks([0, 1], CLASSES)
ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title(f'V3-Max ({test_acc*100:.1f}%)')
plt.savefig('confusion_matrix.png', dpi=150); plt.show()

In [ ]:
# ── 9. Export for DeepShield ─────────────────────────────────────
CKPT = 'deepshield_mobilenetv3.pth'
torch.save({
    'arch':          f'mobilenet_v3_{BACKBONE}',
    'state_dict':    model.state_dict(),
    'classes':       CLASSES,
    'input_size':    IMG,
    'normalize':     {'mean': MEAN, 'std': STD},
    'val_accuracy':  round(history['val_acc'][-1] * 100, 2),
    'robust_val_accuracy': round(max(history['robust_acc']) * 100, 2),
    'tpdn_accuracy': round(tpdn_final * 100, 2),
    'test_accuracy': round(float(test_acc) * 100, 2),
    'trained_on':    f'V3-Max multi-generator: SG1+TPDN/SG2+diffusion, '
                     f'{len(train_samples)} imgs, {EPOCHS} epochs, {BACKBONE}',
}, CKPT)
print(f'Saved {CKPT} — {os.path.getsize(CKPT)/1e6:.1f} MB')
# safety copy to Drive too, then browser download
import shutil
shutil.copy(CKPT, f'{CKPT_DIR}/{CKPT}')  # → Output tab (right panel) to download


## ✅ Done

1. Right panel → **Output** tab → download **`deepshield_mobilenetv3.pth`**, `training_curves.png`, `confusion_matrix.png`.
2. Save the model as `g:deepfakemodelsarchive3_max.pth` first, then copy it over `modelsdeepshield_mobilenetv3.pth` to go live (the backend hot-reloads and reads the arch from the checkpoint).
3. Tell Claude **"v3 aa gaya"** for the TPDN revenge test and a V2-vs-V3 comparison.

> **If the session ends early:** commit/save the notebook version, then start it again — `resume.pth` in `/kaggle/working` lets training pick up from the last finished epoch.